# 1. [calculate] labour productivity
- Save to `working_yearly` new table
**Assets**
- Total assets: book value of all assets
(i.e. intangible and tangible assets, stock, current and non-currents assets)#
- Total liabilities: sum of current liabilities (i.e. loans and short-term debt, creditors and non-current liabilities (i.e. long-term financial liabilities including borrowing from credit institutions and bonds issued).
- Leverage: ratio of total liabilities to total assets.
  
**Income**
- Operating revenue (turnover): sum of net sales, other operating revenues and stock variations.
- Wage bill: renumeration_employees
- Employment: number of employees on the company’s payroll. 
- Negative turnover values. Turnover is defined as the operating revenue in FAME. In a few cases, some companies report negative turnover values. We flag (but keep) those companies reporting negative turnover values.  
   
**Productivity** 
- GVA (Lars): wage bill + EBITDA
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: wage bill / employees
- Use lns

In [4]:
import ibis
from utils.f_0_dirs import get_data_dirs

old_table_name = "fame_yearly_kp"
new_table_name = "working_yearly"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))

# Reference the existing deflated table
fame_yearly = con.table(old_table_name)

# Calculate the new metrics using Ibis lazy evaluation
# We use ibis.ifelse to safely handle natural logarithms of negative or zero GVA
working_yearly = fame_yearly.mutate(
    gva1 = fame_yearly.wages + fame_yearly.ebitda,
    gva2 =  fame_yearly.profit_loss_pretax +
            fame_yearly.interest_paid +
            fame_yearly.depreciation +
            fame_yearly.remuneration_employees
).mutate(
    gva1_per_worker = ibis._.gva1 / fame_yearly.employees,
    gva2_per_worker = ibis._.gva2 / fame_yearly.employees,
    average_wage = fame_yearly.wages / fame_yearly.employees
)

# Verify the final materialized table
table_t_working = ibis.memtable(working_yearly)
print(f"\nSample of {new_table_name}:")
display(table_t_working.sample(0.0001).execute())


Sample of working_yearly:


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda,gva1,gva2,gva1_per_worker,gva2_per_worker,average_wage
0,01365343,2006,False,953.319551,528.835234,6.284452,22,NaN,NaN,NaN,...,44.222720,46.846434,NaN,NaN,-19.897086,328.041318,NaN,14.910969,NaN,15.815382
1,00582503,2006,False,24163.505865,478.118159,8.581604,54,2446.305792,1775.312024,NaN,...,195.893620,52.219577,NaN,39.321,192.401042,1913.056036,2065.677796,35.426964,38.253293,31.863981
2,04049093,2006,False,5647.493171,5683.755690,-253.837633,12,1370.092564,1292.837633,NaN,...,119.185504,463.680590,NaN,NaN,-474.566009,2712.421706,3203.016176,226.035142,266.918015,265.582310
3,03015764,2006,False,9918.587253,-1128.867982,149.779970,26,175.006070,NaN,NaN,...,288.984029,42.449631,NaN,541.000,283.793627,4014.463160,NaN,154.402429,NaN,143.487290
4,03105731,2007,False,5119.708780,6036.500458,781.000707,21,16.369267,1.993089,NaN,...,123.269935,NaN,NaN,308.431,668.510817,1950.726188,NaN,92.891723,NaN,61.057875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,00026306,2023,False,45709.766000,284961.335000,11425.883000,63,6056.331000,4172.624000,4048.983,...,248.024883,206.687403,NaN,NaN,1806.821000,4371.811669,NaN,69.393836,NaN,40.714138
112,SC114228,2023,False,1155.810614,136.404076,-187.491706,51,2.909200,NaN,NaN,...,52.368387,33.776855,NaN,NaN,-184.564843,770.356794,NaN,15.105035,NaN,18.723954
113,02134749,2023,False,NaN,103159.193000,-11472.638000,113,376.118000,NaN,NaN,...,3142.681960,875.321151,1010.0,NaN,-15633.833000,329.668555,8231.638050,2.917421,72.846354,141.269925
114,03928976,2023,False,32961.236000,11318.866000,7152.476000,129,361.572000,322.090000,NaN,...,1271.127527,492.949456,178.0,946.000,8040.821000,18613.915090,19779.598862,144.293915,153.330224,81.961970


In [5]:
working_yearly_skinny = table_t_working.select(
    "registered_number", "year",
    "employees", "fixed_total", "total_assets",
    "average_wage", "gva1", "gva2",
    "gva1_per_worker", "gva2_per_worker"
)

print(f"✅ Inserting columns into new '{new_table_name}' table: {working_yearly_skinny.columns}")
con.create_table(new_table_name, working_yearly_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nHead of {new_table_name}:")
display(final_table.sample(200 / row_count).execute())

✅ Inserting columns into new 'working_yearly' table: ('registered_number', 'year', 'employees', 'fixed_total', 'total_assets', 'average_wage', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker')
✅ Materialized 'working_yearly' table.
📊 Number of rows: 1,102,222
📊 Number of columns: 10

Head of working_yearly:


,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker
0,01553171,2006,513,30889.359636,124013.084977,42.191840,44507.143899,47239.697869,86.758565,92.085181
1,05141592,2006,275,10708.479514,59722.792109,51.087983,5311.504892,7958.894306,19.314563,28.941434
2,02924145,2006,334,2782.754173,37391.386950,43.671696,17056.927621,18173.864194,51.068646,54.412767
3,SC118818,2006,451,20805.226100,22473.301973,15.910467,9458.582457,NaN,20.972467,NaN
4,02762127,2006,11,34.889273,33197.391713,109.500160,4259.145995,3231.103991,387.195090,293.736726
...,...,...,...,...,...,...,...,...,...,...
237,01377957,2023,175,825.553035,4417.960992,30.304253,4643.887614,4294.266032,26.536501,24.538663
238,01223191,2023,31,1288.648842,16579.016495,37.253371,4657.887672,4861.325920,150.254441,156.816965
239,03560591,2023,49,1365.123398,8425.578285,31.710277,1931.829158,NaN,39.425085,NaN
240,06184980,2023,29,NaN,84480.051000,102.880437,19321.807659,NaN,666.269230,NaN


# [calculate] 2. TFP

$$\ln(Y_{it}) = \alpha_i + \gamma_t + \beta_K \ln(K_{it}) + \beta_L \ln(L_{it}) + \varepsilon_{it}$$
- $Y_{it}$: `gva1` or `gva2`
- $K_{it}$: `fixed_total` or `total_assets`
- $L_{it}$: `employees`  
### Capital choice
- `fixed_total` = `tangibles` + `intangibles` + `investments_other`, representing different types of capitals
- `total_assets` = `fixed_total` + `current_assets`, which includes non-productive current_assets (e.g. cash, stock, debtors) and productive current_assets (e.g. stock of raw materials, work in progress and finished goods).

In [6]:
import ibis
from ibis import _
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(str(dirs.db_path))
table_working = con.table("working_yearly")
table_results = table_working.select("registered_number", "year")

# 3. Define the 4 model setups to iterate through
# Varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'tfp1': {'Y': 'gva1', 'K': 'total_assets', 'L': 'employees'},
    'tfp2': {'Y': 'gva2', 'K': 'total_assets', 'L': 'employees'},
    'tfp3': {'Y': 'gva1', 'K': 'fixed_total', 'L': 'employees'}
}

# Dictionary to store the parameter outputs (\beta_K, \beta_L) and model summaries
parameter_tables = {}

for name, mod in models.items():

    # Mutate to dynamically log-transform all columns, adding ln_ prefix to column name
    table_skinny = (
        table_working
        .select(["registered_number", "year"] + list(mod.values()))
        .rename(mod)
    )
    table_start = table_skinny
    for col in ['Y', 'K', 'L']:
        table_filtered = table_start.filter(_[col] > 0)
        table_logged = table_filtered.mutate(**{f'ln_{col}': np.log(_[col]) }) # type: ignore
        table_start = table_logged

    # 1. Execute into a Pandas DataFrame and set the MultiIndex for linearmodels
    df_model = table_start.execute().set_index(['registered_number', 'year'])

    # Define Endogenous (Y) and Exogenous (X) variables
    Y = df_model['ln_Y']
    X = sm.add_constant(df_model[['ln_K', 'ln_L']])
    
    # 2. Estimate the model with Firm and Year Fixed Effects
    mod_ols = PanelOLS(Y, X, entity_effects=True, time_effects=True)
    
    # Fit model with firm-clustered standard errors
    res = mod_ols.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract \beta_K and \beta_L
    beta_K = res.params['ln_K']
    beta_L = res.params['ln_L']
    
    # 3. Calculate firm-year specific TFP (Solow Residual) inside the Ibis pipeline
    # TFP_it = ln(Y_it) - \beta_K*ln(K_it) - \beta_L*ln(L_it)
    table_with_tfp = table_start.mutate(
        **{name: _['ln_Y'] - (beta_K * _['ln_K']) - (beta_L * _['ln_L'])}
    )
    
    # 4. Join the calculated TFP column back to the main dataframe
    # We select only the keys and the new TFP column to avoid duplicating ln_ columns
    table_results = (
        table_results
        .left_join(
            table_with_tfp,
            ["registered_number", "year"],
            rname='{name}_' + name
        )
        .drop("registered_number_" + name, "year_" + name) # Drop duplicate join keys
    )
    
    # 5. Store the results and parameters
    parameter_tables[name] = {
        'beta_K': beta_K,
        'beta_L': beta_L,
        # Safely extract time effects if they exist
        'gamma_t': res.estimated_effects.xs('time_effects', level=1) if 'time_effects' in res.estimated_effects.index.names else None, 
        'summary': res.summary
    }
    print(f"✅ Model '{name}' estimated: beta_K={beta_K:.4f}, beta_L={beta_L:.4f}")

print(f"Panel regressions complete. {len(models)} TFP variants added to the dataframe.")

✅ Model 'tfp1' estimated: beta_K=0.2852, beta_L=0.6174
✅ Model 'tfp2' estimated: beta_K=0.2642, beta_L=0.6114
✅ Model 'tfp3' estimated: beta_K=0.0652, beta_L=0.7194
Panel regressions complete. 3 TFP variants added to the dataframe.


In [7]:
# From the above cell, display the revised table with the new TFP columns and the parameter estimates for each model
table_sample = table_results.sample(0.0001).execute()
display(table_sample)

# Send parameter_tables to a markdown file in dirs.output_dir to easily compare
output_file = dirs.output_dir / f"tfp_2factor_results_{name}.md"
with open(output_file, 'w') as f:
    for name, params in parameter_tables.items():
        # Write all to 1 big markdown file
        # Just write the default display(params) output to the file
        f.write(f"# Parameters for model '{name}'\n\n")
        f.write(f"## Estimated Coefficients\n")
        f.write(f"- beta_K: {params['beta_K']:.6f}\n")
        f.write(f"- beta_L: {params['beta_L']:.6f}\n")
        if params['gamma_t'] is not None:
            f.write(f"\n## Time Effects (gamma_t)\n")
            f.write(params['gamma_t'].to_markdown())
        f.write("\n\n## Model Summary\n")
        f.write(params['summary'].as_text())
        print(f"✅ Parameters for model '{name}' written to {output_file}")

# Verify that that \ln Y = \alpha_i + \gamma_t + \beta_K \ln K + \beta_L \ln L + TFP_it holds for this sample
name, mod = list(models.items())[0]
params = parameter_tables[name]
for row in table_sample.itertuples():
    # Get TFP, L, K
    tfp = row._asdict()[name]
    ln_K = row.ln_K
    ln_L = row.ln_L
    ln_Y = row.ln_Y
    if any(np.isnan([tfp, ln_K, ln_L, ln_Y])):
        print(f"Skipping row {row.Index} due to NaN values.")
        continue
    ln_Y_calc = tfp + params['beta_K'] * ln_K + params['beta_L'] * ln_L
    diff = ln_Y - ln_Y_calc

    assert np.isclose(diff, 0, atol=1e-6), (
        f"TFP calculation check failed for row {row.Index}!  \
        Expected ln_Y: {ln_Y:.6f}, Calculated ln_Y: {ln_Y_calc:.6f}, Difference: {diff:.6e}"
    )

# Count number of tfp1 and tfp2 observations in the results table (as a percentage of total rows)
# Output as a pd dataframe
total_rows = table_results.count().execute()
tfp1_count = table_results.filter(_['tfp1'].isnull() == False).count().execute()
tfp2_count = table_results.filter(_['tfp2'].isnull() == False).count().execute()
tfp3_count = table_results.filter(_['tfp3'].isnull() == False).count().execute()
tfp_counts = pd.DataFrame({
    'TFP Variant': ['tfp1', 'tfp2', 'tfp3'],
    'Count': [tfp1_count, tfp2_count, tfp3_count]
})
tfp_counts['Percentage'] = tfp_counts['Count'] / total_rows * 100
print(f"\nTFP Counts and Percentages (out of {total_rows:,} total rows):")
display(tfp_counts)

,registered_number,year,Y,K,L,ln_Y,ln_K,ln_L,tfp1,Y_tfp2,...,ln_K_tfp2,ln_L_tfp2,tfp2,Y_tfp3,K_tfp3,L_tfp3,ln_Y_tfp3,ln_K_tfp3,ln_L_tfp3,tfp3
0,01421303,2006,19725.102867,62858.997055,242.0,9.889647,11.048649,5.488938,3.349636,17960.520281,...,11.048649,5.488938,3.520414,19725.102867,28543.045375,242.0,9.889647,10.259169,5.488938,5.272280
1,01063450,2006,17415.509608,52828.245100,325.0,9.765116,10.874801,5.783825,3.092634,21193.873748,...,10.874801,5.783825,3.551583,17415.509608,14049.312278,325.0,9.765116,9.550329,5.783825,4.981795
2,03491759,2007,1058.940900,2120.276339,23.0,6.965025,7.659302,3.135494,2.844675,999.755840,...,7.659302,3.135494,2.966537,1058.940900,1744.257144,23.0,6.965025,7.464084,3.135494,4.222903
3,02280503,2007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,812.273301,...,5.527823,3.044522,3.377692,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NI019950,2008,3644.768003,13046.069701,156.0,8.201048,9.476242,5.049856,2.380594,3149.843135,...,9.476242,5.049856,2.463539,3644.768003,12521.569046,156.0,8.201048,9.435208,5.049856,3.953257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,04768344,2014,3769.991585,4277.841495,48.0,8.234828,8.361204,3.871201,3.460076,NaN,...,NaN,NaN,NaN,3769.991585,26.778351,48.0,8.234828,3.287594,3.871201,5.235591
78,01530263,2014,710.192270,2045.710665,17.0,6.565536,7.623501,2.833213,2.642019,NaN,...,NaN,NaN,NaN,710.192270,509.890589,17.0,6.565536,6.234196,2.833213,4.121025
79,02908288,2010,6964.265745,11886.275766,34.0,8.848547,9.383140,3.526361,3.995217,NaN,...,NaN,NaN,NaN,6964.265745,7446.649025,34.0,8.848547,8.915519,3.526361,5.730647
80,09042249,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


✅ Parameters for model 'tfp1' written to C:\Users\lazyst\Files\ucl\Dissertation\descriptives\output\tfp_2factor_results_tfp3.md
✅ Parameters for model 'tfp2' written to C:\Users\lazyst\Files\ucl\Dissertation\descriptives\output\tfp_2factor_results_tfp3.md
✅ Parameters for model 'tfp3' written to C:\Users\lazyst\Files\ucl\Dissertation\descriptives\output\tfp_2factor_results_tfp3.md
Skipping row 3 due to NaN values.
Skipping row 7 due to NaN values.
Skipping row 9 due to NaN values.
Skipping row 67 due to NaN values.
Skipping row 76 due to NaN values.
Skipping row 80 due to NaN values.

TFP Counts and Percentages (out of 1,102,222 total rows):


,TFP Variant,Count,Percentage
0,tfp1,1020691,92.603033
1,tfp2,592751,53.777823
2,tfp3,983564,89.234655


In [8]:
preferred_model = 'tfp1'
# Add the tfp1 column from this table_results to the working_yearly table in the database
# Rename as 'tfp'
table_with_tfp = (
    table_working
    .left_join(
        table_results.select(['registered_number', 'year', preferred_model]),
        ['registered_number', 'year']
    )
    # Rename the preferred TFP column to 'tfp' for clarity
    .rename(tfp=preferred_model)
    .drop("registered_number_right", "year_right")
)
display(table_with_tfp.sample(0.0001).execute())

,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker,tfp
0,03213873,2006,4126,34753.682853,438755.983308,27.275538,130889.282237,160343.509412,31.723045,38.861733,2.936948
1,03368442,2006,146,21287.612200,28982.824592,58.409284,14363.036433,14522.865888,98.376962,99.471684,3.565194
2,SC044454,2006,210,2085.069615,17172.692904,21.063219,4226.359573,8824.327873,20.125522,42.020609,2.266739
3,02644277,2006,41,2924.650986,8726.654021,54.555342,6373.849467,NaN,155.459743,NaN,3.879182
4,01530915,2006,93,1433.734862,11572.314205,50.382323,8201.919884,NaN,88.192687,NaN,3.545201
...,...,...,...,...,...,...,...,...,...,...,...
123,SC185346,2023,79,1854.843580,10460.109642,28.213969,3368.485990,NaN,42.639063,NaN,2.784842
124,00500574,2023,176,4532.118000,18893.176000,24.385590,5571.911919,3925.115411,31.658590,22.301792,2.624947
125,11963500,2023,266,30616.213000,37316.724000,42.471930,9677.732437,NaN,36.382453,NaN,2.727921
126,07198322,2023,257,5015.648859,14749.507891,31.797502,8506.796490,9020.541368,33.100375,35.099383,2.884958


In [ ]:
overwrite = True
if overwrite:
    # Safe overwrite of the working_yearly table with the new tfp column
    con.create_table("working_yearly_temp", table_with_tfp)
    con.create_table("working_yearly", con.table("working_yearly_temp"), overwrite=True)
    con.drop_table("working_yearly_temp")

    # Verify schema and rows of the final working_yearly table
    final_table = con.table("working_yearly")
    row_count = final_table.count().execute()
    col_count = len(final_table.columns)
    tfp_col_exists = 'tfp' in final_table.columns
    tfp_nan_count = final_table.filter(_['tfp'].isnull() == True).count().execute() if tfp_col_exists else None
    print(f"\nFinal 'working_yearly' table verification:"
        f"\n- Rows: {row_count:,}"
        f"\n- Columns: {col_count:,}"
        f"\n- TFP column exists: {tfp_col_exists}"
        f"\n- TFP NaN count: {tfp_nan_count:,}")


Final 'working_yearly' table verification:
- Rows: 1102222
- Columns: 11
- TFP column exists: True
- TFP NaN count: 81531
